# SQL — Question Patterns Reference

**Companion to:** `reference_sql_syntax.ipynb` — function names, argument order, dialect differences

**Purpose:** recognize problem types and reach for the right template. Open this when you're mid-problem and need to identify the pattern.

## Quick Index

| Section | Pattern |
| :--- | :--- |
| §1 | Deduplication patterns |
| §2 | Ranking & Top N per group |
| §3 | Pivot & transpose |
| §4 | Gaps & Islands — streak detection |
| §5 | Session grouping |
| §6 | Product metrics: DAU/WAU/MAU, cohort, funnel |
| §7 | Growth metrics: WoW, MoM, rolling active users |
| §8 | Recursive patterns |
| §9 | Advanced aggregation patterns |

**§8 sub-sections:**
| | |
| :--- | :--- |
| §8a | Org chart top-down traversal |
| §8b | Find all ancestors (bottom-up) |
| §8c | Date series generation |
| §8d | Number series generation |
| §8e | Multi-level hierarchy with path & aggregation |
| §8f | Cycle detection |
| §8g | Cycle length measurement |
| §8h | Shortest path between nodes (BFS) |
| §8i | Running total / cost accumulation through a chain |
| §8j | Flatten a linked list |
| — | Decision Guide |

---
## When to Use

| Signal words | Pattern |
| :--- | :--- |
| "remove duplicates", "keep only one" | §1 — Deduplication |
| "top N per group", "highest salary per dept" | §2 — Ranking |
| "pivot", "rows to columns", "each X as a column" | §3 — Pivot |
| "consecutive days", "streak", "unbroken sequence" | §4 — Gaps & Islands |
| "session", "group events within N minutes" | §5 — Session grouping |
| "DAU", "retention", "cohort", "funnel" | §6 — Product metrics |
| "week over week", "growth rate", "rolling users" | §7 — Growth metrics |
| "hierarchy", "org chart", "all ancestors" | §8 — Recursive CTE |
| "cycle", "loop in graph", "circular reference" | §8f — Cycle detection |
| "cycle length", "loop size" | §8g — Cycle length |
| "shortest path", "minimum hops", "fewest steps" | §8h — BFS shortest path |
| "bill of materials", "total cost through chain", "running sum down hierarchy" | §8i — Chain accumulation |
| "linked list", "next pointer", "kth node", "flatten chain" | §8j — Linked list flatten |
| "best match", "pair rows", "all conditions met" | §9 — Advanced aggregation |

---
## §1 — Deduplication Patterns

**Signal:** "remove duplicates", "keep the earliest/latest record", "unique users only"

### §1a — DISTINCT (fully identical rows)

```sql
-- Remove fully duplicate rows across all selected columns
SELECT DISTINCT user_id, event_date FROM events;
```

### §1b — GROUP BY as deduplication

```sql
-- Deduplicate and aggregate simultaneously
SELECT user_id, COUNT(*) AS login_count, MAX(login_date) AS last_login
FROM logins
GROUP BY user_id;
```

### §1c — ROW_NUMBER() — keep specific occurrence

```sql
-- Keep the FIRST record per user (earliest date)
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at ASC) AS rn
  FROM orders
)
SELECT * FROM ranked WHERE rn = 1;

-- Keep the LAST record per user (latest date)
WITH ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at DESC) AS rn
  FROM orders
)
SELECT * FROM ranked WHERE rn = 1;
```

### §1d — NOT EXISTS / NOT IN — remove rows with a match elsewhere

```sql
-- Keep users who have never made a purchase
SELECT * FROM users u
WHERE NOT EXISTS (
  SELECT 1 FROM orders o WHERE o.user_id = u.id
);
```

### §1e — DELETE duplicates, keep one

```sql
-- Keep row with smallest id, delete the rest
DELETE p1 FROM Person p1
JOIN Person p2
  ON p1.email = p2.email
 AND p1.id > p2.id;
```

**Decision — which deduplication method:**

| Method | Use when |
| :--- | :--- |
| `DISTINCT` | All selected columns define uniqueness |
| `GROUP BY` | Need to deduplicate AND aggregate simultaneously |
| `ROW_NUMBER()` | Need to keep a specific occurrence (first, last, nth) |
| `NOT EXISTS` | Remove rows that have any match in another table |
| `DELETE + JOIN` | Physically remove duplicates from the table |

**Common mistakes:**
- Using `DISTINCT` when you need the first or last — DISTINCT doesn't control which row is kept
- Using `NOT IN` when the subquery can return NULLs — always prefer `NOT EXISTS` for safety
- Forgetting `PARTITION BY` in `ROW_NUMBER()` — without it, numbering is global not per-group

---
## §2 — Ranking & Top N per Group

**Signal:** "highest salary per department", "top 3 products per category", "second highest"

### §2a — Top N per group (ROW_NUMBER)

```sql
-- Top 3 salaries per department — no ties
WITH ranked AS (
  SELECT dept, emp_id, salary,
    ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rn
  FROM Employee
)
SELECT dept, emp_id, salary FROM ranked WHERE rn <= 3;
```

### §2b — Top N per group with ties (DENSE_RANK)

```sql
-- Top 3 salary levels per department — ties included
WITH ranked AS (
  SELECT dept, emp_id, salary,
    DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dr
  FROM Employee
)
SELECT dept, emp_id, salary FROM ranked WHERE dr <= 3;
```

### §2c — Nth highest (single value)

```sql
-- Nth highest salary — handles case where fewer than N salaries exist
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  -- MySQL requires LIMIT/OFFSET arguments to be literal integers or variables,
  -- never expressions. Writing "OFFSET N-1" directly is a syntax error, so the
  -- offset must be computed into the variable first.
  SET N = N - 1;
  RETURN (
    SELECT DISTINCT salary FROM Employee
    ORDER BY salary DESC
    LIMIT 1 OFFSET N
  );
END;

-- Or with DENSE_RANK
WITH ranked AS (
  SELECT salary,
    DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
  FROM Employee
)
SELECT MAX(salary) FROM ranked WHERE dr = N;
-- MAX handles the case where N rank doesn't exist — returns NULL
```

### §2d — Median

```sql
-- Median using ROW_NUMBER (works in MySQL)
WITH counted AS (
  SELECT id, salary, COUNT(*) OVER () AS total,
    ROW_NUMBER() OVER (ORDER BY salary) AS rn
  FROM Employee
)
SELECT AVG(salary) AS median
FROM counted
WHERE rn IN (FLOOR((total+1)/2), CEIL((total+1)/2));
```

**Decision — ROW_NUMBER vs DENSE_RANK for Top N:**
| Function | Use when |
| :--- | :--- |
| `ROW_NUMBER()` | Want exactly N rows, ties broken arbitrarily |
| `DENSE_RANK()` | Want top N distinct values, ties all included |
| `LIMIT + OFFSET` | Single value without window function support |

**Common mistakes:**
- Using `RANK()` for top N — gaps after ties mean rank 2 may not exist
- Forgetting `PARTITION BY` — without it, ranking is global not per-group
- Using `LIMIT N` directly without `DISTINCT` — may return duplicate salary values
- `LIMIT + OFFSET` fails silently for some kth-rank problems — prefer `DENSE_RANK` as the reliable fallback

---
## §3 — Pivot & Transpose

**Signal:** "each month as a column", "show X and Y side by side", "convert rows to columns"

### §3a — Pivot with CASE WHEN (most common)

```sql
-- Pivot: one row per product, one column per month
SELECT
  product_id,
  SUM(CASE WHEN MONTH(sale_date) = 1 THEN amount ELSE 0 END) AS Jan,
  SUM(CASE WHEN MONTH(sale_date) = 2 THEN amount ELSE 0 END) AS Feb,
  SUM(CASE WHEN MONTH(sale_date) = 3 THEN amount ELSE 0 END) AS Mar
FROM sales
GROUP BY product_id;
```

### §3b — Transpose (rows to columns, fixed columns)

```sql
-- LC 1179: Department table with Department and Revenue per quarter
SELECT
  SUM(CASE WHEN quarter = 'Q1' THEN revenue END) AS Q1_Revenue,
  SUM(CASE WHEN quarter = 'Q2' THEN revenue END) AS Q2_Revenue,
  SUM(CASE WHEN quarter = 'Q3' THEN revenue END) AS Q3_Revenue,
  SUM(CASE WHEN quarter = 'Q4' THEN revenue END) AS Q4_Revenue
FROM Department;
```

### §3c — Dynamic pivot (variable number of columns)

```sql
-- Not directly supported in MySQL — requires dynamic SQL
-- Use GROUP_CONCAT to build the CASE WHEN list dynamically
SET @sql = NULL;
SELECT GROUP_CONCAT(
  DISTINCT CONCAT(
    'SUM(CASE WHEN month = ', month, ' THEN amount ELSE 0 END) AS m', month
  )
) INTO @sql FROM sales;

SET @sql = CONCAT('SELECT product_id, ', @sql, ' FROM sales GROUP BY product_id');
PREPARE stmt FROM @sql;
EXECUTE stmt;
DEALLOCATE PREPARE stmt;
-- GROUP_CONCAT silently truncates at group_concat_max_len (1024 bytes by default),
-- producing an incomplete column list. Raise it before building @sql:
--   SET SESSION group_concat_max_len = 1000000;
```

### §3d — Top N + Pivot combined

**Signal:** "show top N items per group as separate columns"

When you need to pivot while also selecting only the top N items per group, combine `ROW_NUMBER()` with `GROUP BY` + `MAX`/`MIN`:

```sql
-- Align rows with their category rank, then collapse to one row per group
WITH ranked AS (
  SELECT
    dept,
    salary,
    ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rn
  FROM Employee
)
SELECT
  dept,
  MAX(CASE WHEN rn = 1 THEN salary END) AS top1_salary,
  MAX(CASE WHEN rn = 2 THEN salary END) AS top2_salary,
  MAX(CASE WHEN rn = 3 THEN salary END) AS top3_salary
FROM ranked
GROUP BY dept;
-- MAX extracts the single non-null value per cell; COALESCE fills blanks if needed
```

```sql
-- Fill missing pivot positions with COALESCE
SELECT
  dept,
  COALESCE(MAX(CASE WHEN rn = 1 THEN salary END), 'N/A') AS top1_salary,
  COALESCE(MAX(CASE WHEN rn = 2 THEN salary END), 'N/A') AS top2_salary
FROM ranked
GROUP BY dept;
```

**Common mistakes:**
- Forgetting `ELSE 0` in `SUM(CASE WHEN ...)` — non-matching rows contribute NULL, not 0
- Using `MAX` vs `SUM` in pivot — use `MAX` when each cell has only one value; `SUM` when aggregating multiple rows
- Not grouping by the row identifier — always `GROUP BY` the column that defines each output row

---
## §4 — Gaps & Islands — Streak Detection

**Signal:** "consecutive days", "longest streak", "unbroken sequence", "find gaps in dates"

### §4a — Date streak detection (row number subtraction trick)

```sql
-- Trick: date shifted back by its row number = constant within a consecutive streak
-- If dates are consecutive daily, the shifted value stays the same
WITH numbered AS (
  SELECT user_id, activity_date,
    DATE_SUB(activity_date, INTERVAL ROW_NUMBER() OVER (
      PARTITION BY user_id ORDER BY activity_date
    ) DAY) AS grp                      -- same value = same streak
  FROM activity
)
SELECT
  user_id,
  MIN(activity_date) AS streak_start,
  MAX(activity_date) AS streak_end,
  COUNT(*)           AS streak_length
FROM numbered
GROUP BY user_id, grp
ORDER BY user_id, streak_start;
```

**Postgres / standard SQL only — do not use this form in MySQL:**

In Postgres, `date - integer` returns a date, so the plain subtraction works. MySQL
instead coerces the date to a `YYYYMMDD` **number**, so the arithmetic silently breaks
at every month boundary.

```sql
-- Postgres: correct.  MySQL: WRONG.
WITH numbered AS (
  SELECT user_id, activity_date,
    activity_date - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY activity_date) AS grp
  FROM activity
)
SELECT user_id, grp, MIN(activity_date), MAX(activity_date), COUNT(*) AS streak_len
FROM numbered
GROUP BY user_id, grp;
```

Five consecutive days spanning a month boundary, evaluated the way MySQL does it:

| activity_date | rn | `date - rn` in MySQL | `DATE_SUB(...)` |
| :--- | :--- | :--- | :--- |
| 2024-02-27 | 1 | 20240226 | 2024-02-26 |
| 2024-02-28 | 2 | 20240226 | 2024-02-26 |
| 2024-02-29 | 3 | 20240226 | 2024-02-26 |
| 2024-03-01 | 4 | **20240297** | 2024-02-26 |
| 2024-03-02 | 5 | **20240297** | 2024-02-26 |

The `grp` value changes mid-streak, so one 5-day streak is reported as two.

### §4b — Longest streak per user

```sql
WITH numbered AS (
  SELECT user_id, activity_date,
    DATE_SUB(activity_date, INTERVAL ROW_NUMBER() OVER (
      PARTITION BY user_id ORDER BY activity_date
    ) DAY) AS grp
  FROM activity
),
streak_lengths AS (
  SELECT user_id, grp, COUNT(*) AS streak_len
  FROM numbered
  GROUP BY user_id, grp
)
SELECT user_id, MAX(streak_len) AS longest_streak
FROM streak_lengths
GROUP BY user_id;
```

### §4c — Integer / seat number gaps (same trick)

```sql
-- Find consecutive seat ranges
WITH numbered AS (
  SELECT seat_id,
    seat_id - ROW_NUMBER() OVER (ORDER BY seat_id) AS grp
  FROM available_seats
)
SELECT MIN(seat_id) AS range_start, MAX(seat_id) AS range_end, COUNT(*) AS seats
FROM numbered
GROUP BY grp
HAVING COUNT(*) >= 2;                  -- only groups of 2+ consecutive seats
```

### §4d — Flag-then-sum (boundary flag approach)

**Use when:** the row-subtraction trick is hard to apply, or when you need to detect boundaries based on a condition (not just consecutive integers/dates).

```sql
-- Step 1: flag each row as 1 if it starts a new group, 0 otherwise
WITH flagged AS (
  SELECT
    user_id,
    activity_date,
    CASE
      WHEN activity_date != DATE_ADD(
        LAG(activity_date) OVER (PARTITION BY user_id ORDER BY activity_date),
        INTERVAL 1 DAY
      ) THEN 1 ELSE 0
    END AS is_new_group
  FROM activity
),
-- Step 2: cumulative sum of flags = group ID
grouped AS (
  SELECT
    user_id,
    activity_date,
    SUM(is_new_group) OVER (PARTITION BY user_id ORDER BY activity_date) AS grp_id
  FROM flagged
)
SELECT user_id, grp_id, MIN(activity_date), MAX(activity_date), COUNT(*) AS streak_len
FROM grouped
GROUP BY user_id, grp_id;
```

**Flag direction matters:** setting the flag to 1 at the *start* of a new group (as above) vs 1 at the *end* of a group gives different cumulative sums. Always verify which boundary you're flagging.

### §4e — LAG comparison (consecutive-check only)

**Use when:** you just need to know *whether* a pair of rows is consecutive — no group ID needed.

```sql
-- Check if each day is consecutive with the previous
SELECT
  user_id,
  activity_date,
  LAG(activity_date) OVER (PARTITION BY user_id ORDER BY activity_date) AS prev_date,
  TIMESTAMPDIFF(DAY,
    LAG(activity_date) OVER (PARTITION BY user_id ORDER BY activity_date),
    activity_date
  ) AS gap_days
FROM activity;
-- gap_days = 1 means consecutive; > 1 means a gap; NULL means first row
```

### §4f — Month-based consecutiveness (PERIOD_DIFF)

**Use when:** you want to detect consecutive months instead of consecutive days.

```sql
WITH ordered AS (
  SELECT
    user_id,
    DATE_FORMAT(activity_date, '%Y%m') AS yr_month
  FROM (SELECT DISTINCT user_id, DATE_FORMAT(activity_date, '%Y-%m-01') AS activity_date FROM activity) deduped
),
grouped AS (
  SELECT
    user_id,
    yr_month,
    -- PERIOD_DIFF returns months between two YYYYMM values
    PERIOD_DIFF(yr_month,
      LAG(yr_month) OVER (PARTITION BY user_id ORDER BY yr_month)
    ) AS month_gap
  FROM ordered
)
-- month_gap = 1 means consecutive months; group using flag-then-sum from §4d
SELECT * FROM grouped;
```

### §4g — Finding missing values in a sequence

```sql
-- Find missing dates using date series + LEFT JOIN
WITH RECURSIVE date_series AS (
  SELECT MIN(activity_date) AS dt FROM activity
  UNION ALL
  SELECT DATE_ADD(dt, INTERVAL 1 DAY)
  FROM date_series
  WHERE dt < (SELECT MAX(activity_date) FROM activity)
)
SELECT d.dt AS missing_date
FROM date_series d
LEFT JOIN activity a ON d.dt = a.activity_date
WHERE a.activity_date IS NULL;
-- MySQL stops at cte_max_recursion_depth (1000 by default), so a range longer
-- than ~3 years errors out. Raise it first:
--   SET SESSION cte_max_recursion_depth = 100000;
```

**Decision — which Gaps & Islands approach:**
| Method | Use when |
| :--- | :--- |
| `DATE_SUB(..., INTERVAL ROW_NUMBER() DAY)` | Consecutive daily dates in MySQL — the correct form |
| Row subtraction (`date - ROW_NUMBER()`) | Integer sequences anywhere; **dates in Postgres only** |
| Flag-then-sum | Condition-based boundaries, or non-fixed step sizes |
| LAG comparison | Only need to identify gaps — no group assignment needed |
| `PERIOD_DIFF` | Consecutive months (not days) |
| Recursive CTE + LEFT JOIN | Find missing values in a sequence |

**Common mistakes:**
- Using `date - ROW_NUMBER()` on dates in MySQL — MySQL converts the date to a `YYYYMMDD` number, so streaks split at month boundaries; use `DATE_SUB(date, INTERVAL ROW_NUMBER() ... DAY)`
- Using `RANK()` instead of `ROW_NUMBER()` — ties in dates break the subtraction trick; always use `ROW_NUMBER()`
- Forgetting `PARTITION BY user_id` — groups streaks across all users instead of per user
- Not deduplicating dates first — if a user has multiple events on the same date, duplicate dates break the row-subtraction trick
- Setting the flag at the wrong boundary (start vs end of group) in the flag-then-sum approach — verify which side you're marking

---
## §5 — Session Grouping

**Signal:** "group events into sessions", "events within N minutes of each other belong to the same session"

This is a generalized Gaps & Islands variant — instead of consecutive dates, you group events where the gap to the previous event is below a threshold.

### §5a — LAG-based session grouping

```sql
-- Step 1: flag new session start (gap > threshold)
WITH flagged AS (
  SELECT
    user_id, event_time,
    CASE
      WHEN TIMESTAMPDIFF(
        MINUTE,
        LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time),
        event_time
      ) > 30                           -- 30-minute session gap threshold
      OR LAG(event_time) OVER (PARTITION BY user_id ORDER BY event_time) IS NULL
      THEN 1 ELSE 0
    END AS is_new_session
  FROM events
),
-- Step 2: assign session ID using cumulative sum of new session flags
sessions AS (
  SELECT
    user_id, event_time,
    SUM(is_new_session) OVER (
      PARTITION BY user_id ORDER BY event_time
    ) AS session_id
  FROM flagged
)
-- Step 3: aggregate per session
SELECT
  user_id,
  session_id,
  MIN(event_time)  AS session_start,
  MAX(event_time)  AS session_end,
  COUNT(*)         AS events_in_session,
  TIMESTAMPDIFF(MINUTE, MIN(event_time), MAX(event_time)) AS session_duration_min
FROM sessions
GROUP BY user_id, session_id
ORDER BY user_id, session_id;
```

**Decision — Gaps & Islands vs Session Grouping:**
| Pattern | Use when |
| :--- | :--- |
| Row number subtraction | Consecutive dates or integers — fixed step size |
| LAG + cumulative SUM | Variable time gaps — group by threshold |

**Common mistakes:**
- Forgetting `OR LAG(...) IS NULL` for the first event — first row has no previous event and should always start a new session
- Using `DATEDIFF` instead of `TIMESTAMPDIFF` — `DATEDIFF` only returns days; use `TIMESTAMPDIFF(MINUTE, ...)` for sub-day gaps
- Applying the session flag without cumulative SUM — a flag alone doesn't assign a unique session ID

---
## §6 — Product Metrics

**Signal:** "daily active users", "retention rate", "cohort", "conversion funnel"

### §6a — DAU / WAU / MAU

```sql
-- Use "= CURRENT_DATE" for a single day, and ">" (not ">=") for N-day windows:
-- ">= CURRENT_DATE - INTERVAL 7 DAY" covers 8 distinct dates, not 7.
SELECT
  COUNT(DISTINCT CASE
    WHEN event_date = CURRENT_DATE                   THEN user_id END) AS DAU,
  COUNT(DISTINCT CASE
    WHEN event_date > CURRENT_DATE - INTERVAL 7 DAY  THEN user_id END) AS WAU,
  COUNT(DISTINCT CASE
    WHEN event_date > CURRENT_DATE - INTERVAL 30 DAY THEN user_id END) AS MAU
FROM events;
```

### §6b — Cohort retention

```sql
-- Assign each user to their cohort (first activity date)
WITH cohort AS (
  SELECT user_id, MIN(DATE(event_ts)) AS cohort_date
  FROM events
  GROUP BY user_id
),
-- Calculate week number relative to cohort date
activity AS (
  SELECT
    e.user_id,
    c.cohort_date,
    FLOOR(DATEDIFF(DATE(e.event_ts), c.cohort_date) / 7) AS week_num
  FROM events e
  JOIN cohort c ON e.user_id = c.user_id
)
SELECT
  cohort_date,
  week_num,
  COUNT(DISTINCT user_id) AS retained_users
FROM activity
GROUP BY cohort_date, week_num
ORDER BY cohort_date, week_num;
```

### §6c — Funnel analysis

```sql
-- Count distinct users at each step
SELECT
  COUNT(DISTINCT CASE WHEN step = 'view'     THEN user_id END) AS step1_view,
  COUNT(DISTINCT CASE WHEN step = 'cart'     THEN user_id END) AS step2_cart,
  COUNT(DISTINCT CASE WHEN step = 'checkout' THEN user_id END) AS step3_checkout
FROM events;

-- With conversion rates
WITH steps AS (
  SELECT
    COUNT(DISTINCT CASE WHEN step = 'view'     THEN user_id END) AS s1,
    COUNT(DISTINCT CASE WHEN step = 'cart'     THEN user_id END) AS s2,
    COUNT(DISTINCT CASE WHEN step = 'checkout' THEN user_id END) AS s3
  FROM events
)
SELECT
  s1 AS view_users,
  s2 AS cart_users,
  s3 AS checkout_users,
  ROUND(s2 * 100.0 / NULLIF(s1, 0), 2) AS view_to_cart_pct,
  ROUND(s3 * 100.0 / NULLIF(s2, 0), 2) AS cart_to_checkout_pct
FROM steps;
```

**Funnel with ordered steps (user must complete steps in sequence):**
When users must hit steps in order, use `JOIN ... ON ... AND` to filter — not `WHERE` — to preserve all step-1 users in the result.

```sql
-- Keep all step-1 users; join step-2 only where it follows step-1
SELECT
  s1.user_id,
  s1.event_time AS step1_time,
  s2.event_time AS step2_time
FROM events s1
LEFT JOIN events s2
  ON s1.user_id = s2.user_id
  AND s2.step = 'cart'
  AND s2.event_time > s1.event_time   -- step 2 must come after step 1
WHERE s1.step = 'view';
-- CAUTION: this fans out. A user with 3 views and 2 later carts produces 6 rows,
-- so never COUNT(*) this result directly.

-- Counting correctly: one row per user, then aggregate
SELECT
  COUNT(DISTINCT s1.user_id) AS step1_users,
  COUNT(DISTINCT s2.user_id) AS step2_users,
  ROUND(COUNT(DISTINCT s2.user_id) * 100.0
        / NULLIF(COUNT(DISTINCT s1.user_id), 0), 2) AS view_to_cart_pct
FROM events s1
LEFT JOIN events s2
  ON s1.user_id = s2.user_id
  AND s2.step = 'cart'
  AND s2.event_time > s1.event_time
WHERE s1.step = 'view';
```

**Common mistakes:**
- Funnel: `WHERE step = 'view'` instead of `CASE WHEN` — collapses to only view-step rows
- Cohort: not deduplicating events per day — use `COUNT(DISTINCT user_id)` not `COUNT(*)`
- Division by zero in conversion rates — always wrap denominator in `NULLIF(..., 0)`
- Ordered funnel: using `WHERE` instead of `ON + AND` — loses users who didn't reach step 2
- Ordered funnel: counting the joined rows directly — the self-join fans out to one row per (step1, step2) pair; always `COUNT(DISTINCT user_id)`

---
## §7 — Growth Metrics

**Signal:** "week-over-week change", "month-over-month growth", "rolling 7-day active users"

### §7a — Week-over-week / Month-over-month growth

```sql
-- Weekly active users with WoW growth rate
WITH weekly AS (
  SELECT
    YEARWEEK(event_date)              AS yr_week,
    COUNT(DISTINCT user_id)           AS wau
  FROM events
  GROUP BY YEARWEEK(event_date)
)
SELECT
  yr_week,
  wau,
  LAG(wau) OVER (ORDER BY yr_week)   AS prev_week_wau,
  ROUND(
    (wau - LAG(wau) OVER (ORDER BY yr_week)) * 100.0
    / NULLIF(LAG(wau) OVER (ORDER BY yr_week), 0),
    2
  )                                   AS wow_growth_pct
FROM weekly
ORDER BY yr_week;

-- Month-over-month: replace YEARWEEK with DATE_FORMAT(date, '%Y-%m')
-- CAUTION: LAG compares against the previous week *present in the data*. A week with
-- no events is absent from the CTE, so growth is silently measured against an older
-- week. If gaps are possible, LEFT JOIN a complete week series before applying LAG.
```

### §7b — Rolling N-day active users (sliding window)

```sql
-- Rolling 7-day distinct active users for each date
-- Different from MAU: window slides daily, not fixed from a start date
WITH daily AS (
  SELECT DISTINCT user_id, DATE(event_ts) AS event_date
  FROM events
)
SELECT
  d1.event_date,
  COUNT(DISTINCT d2.user_id) AS rolling_7d_users
FROM
  (SELECT DISTINCT event_date FROM daily) d1
  JOIN daily d2
    ON d2.event_date BETWEEN DATE_SUB(d1.event_date, INTERVAL 6 DAY)
                         AND d1.event_date
GROUP BY d1.event_date
ORDER BY d1.event_date;
```

**Self-join for pair-based aggregation:**
Self-joins are useful when you need to aggregate over pairs of rows from the same table (e.g., compare distances between all point pairs, or match records by a condition).

```sql
-- Example: aggregate across all pairs satisfying a condition
SELECT
  a.id AS id1, b.id AS id2,
  SQRT(POW(a.x - b.x, 2) + POW(a.y - b.y, 2)) AS distance
FROM points a
JOIN points b ON a.id < b.id   -- avoid counting (a,b) and (b,a) separately
ORDER BY distance;
```

### §7c — Day-1 / Day-7 retention

```sql
-- For each install date cohort, what fraction returned on day 1 and day 7?
WITH installs AS (
  SELECT player_id, MIN(event_date) AS install_date
  FROM Activity
  GROUP BY player_id
)
SELECT
  i.install_date,
  COUNT(DISTINCT i.player_id) AS cohort_size,
  ROUND(
    COUNT(DISTINCT CASE
      WHEN a.event_date = DATE_ADD(i.install_date, INTERVAL 1 DAY)
      THEN a.player_id END
    ) * 1.0 / COUNT(DISTINCT i.player_id),
    2
  ) AS day1_retention,
  ROUND(
    COUNT(DISTINCT CASE
      WHEN a.event_date = DATE_ADD(i.install_date, INTERVAL 7 DAY)
      THEN a.player_id END
    ) * 1.0 / COUNT(DISTINCT i.player_id),
    2
  ) AS day7_retention
FROM installs i
LEFT JOIN Activity a ON i.player_id = a.player_id
GROUP BY i.install_date
ORDER BY i.install_date;
-- For a single overall retention figure instead of per-cohort rows,
-- drop both the GROUP BY and i.install_date from the SELECT.
```

**Decision — fixed window vs rolling window:**
| Metric | Window type | Use when |
| :--- | :--- | :--- |
| MAU | Fixed 30-day from today | Snapshot of recent activity |
| Rolling 7d users | Sliding 7-day per date | Trend over time, each date gets its own window |

**Common mistakes:**
- Rolling window via self-join without deduplication — use `DISTINCT user_id` before joining
- WoW growth: using `WEEK()` instead of `YEARWEEK()` — `WEEK()` resets at year boundary causing wrong week 1 pairing
- Day-N retention: off-by-one in `INTERVAL` — day 1 retention = returned exactly 1 day after install
- Using `LAG`/`LEAD` in the same query as `GROUP BY` without a CTE — always pre-aggregate in a CTE, then apply `LAG` in the outer query

---
## §8 — Recursive Patterns

**Signal:** "hierarchy", "org chart", "all managers of", "generate date series", "find all ancestors"

### §8a — Org chart top-down traversal

```sql
WITH RECURSIVE hierarchy AS (
  -- Anchor: root nodes
  SELECT id, name, manager_id, 0 AS depth
  FROM employees
  WHERE manager_id IS NULL

  UNION ALL

  -- Recursive: one level deeper each iteration
  SELECT e.id, e.name, e.manager_id, h.depth + 1
  FROM employees e
  JOIN hierarchy h ON e.manager_id = h.id
)
SELECT * FROM hierarchy ORDER BY depth, id;
```

### §8b — Find all ancestors (bottom-up)

```sql
WITH RECURSIVE ancestors AS (
  -- Anchor: start from a specific node
  SELECT id, name, manager_id
  FROM employees
  WHERE id = 42

  UNION ALL

  -- Recursive: walk up to parent
  SELECT e.id, e.name, e.manager_id
  FROM employees e
  JOIN ancestors a ON e.id = a.manager_id
)
SELECT * FROM ancestors;
```

### §8c — Date series generation

```sql
-- Generate every date in a range
-- Use case: LEFT JOIN to show zero-activity days
WITH RECURSIVE date_series AS (
  SELECT '2024-01-01' AS dt
  UNION ALL
  SELECT DATE_ADD(dt, INTERVAL 1 DAY)
  FROM date_series
  WHERE dt < '2024-01-31'
)
SELECT d.dt, COALESCE(COUNT(e.user_id), 0) AS event_count
FROM date_series d
LEFT JOIN events e ON DATE(e.event_ts) = d.dt
GROUP BY d.dt
ORDER BY d.dt;
```

### §8d — Number series generation

```sql
-- Generate integers 1 to N
WITH RECURSIVE numbers AS (
  SELECT 1 AS n
  UNION ALL
  SELECT n + 1 FROM numbers WHERE n < 100
)
SELECT * FROM numbers;
```

### §8e — Multi-level hierarchy (set up relationships before aggregating)

**Signal:** "find all descendants", "path from root to node", "total reports under a manager"

For hierarchy problems, first build the full relationship structure in one or more CTEs (level, descendants, path), then do any processing or aggregation on top.

```sql
-- Step 1: build full hierarchy with path
WITH RECURSIVE org AS (
  SELECT id, name, manager_id, CAST(id AS CHAR(200)) AS path, 0 AS lvl
  FROM employees
  WHERE manager_id IS NULL
  UNION ALL
  SELECT e.id, e.name, e.manager_id,
         CONCAT(o.path, '->', e.id), o.lvl + 1
  FROM employees e
  JOIN org o ON e.manager_id = o.id
  WHERE o.lvl < 10                  -- safety limit to prevent runaway recursion
)
-- Step 2: aggregate on top of the hierarchy CTE
SELECT manager_id, COUNT(*) AS total_reports
FROM org
WHERE manager_id IS NOT NULL
GROUP BY manager_id;
```

**Common mistakes:**
- Using `UNION` instead of `UNION ALL` — deduplication breaks the recursion
- Missing termination condition — causes infinite recursion
- Column types in anchor vs recursive step must match exactly
- Writing `RECURSIVE WITH` — correct syntax is `WITH RECURSIVE`

---
### §8f — Cycle detection

**Signal:** "cycle", "loop in graph", "circular reference", "detect if a path revisits a node"

A cycle exists when following `next` pointers eventually brings you back to a node you already visited. Without a guard, the recursive CTE loops forever. The fix is to carry a path string of visited node IDs and stop recursing when the next node already appears in it.

```sql
-- Table: nodes(id, next_id)  -- next_id points to the next node; NULL = end of chain
WITH RECURSIVE traversal AS (
  -- Anchor: start from every node (or a specific start node)
  SELECT
    id                          AS start_id,
    id                          AS current_id,
    next_id,
    CAST(id AS CHAR(1000))      AS visited_path,  -- comma-separated list of visited IDs
    0                           AS depth,
    FALSE                       AS is_cycle
  FROM nodes

  UNION ALL

  SELECT
    t.start_id,
    n.id                        AS current_id,
    n.next_id,
    CONCAT(t.visited_path, ',', n.id),
    t.depth + 1,
    -- cycle detected when the next node is already in the visited path
    FIND_IN_SET(n.id, t.visited_path) > 0
  FROM nodes n
  JOIN traversal t ON n.id = t.next_id
  WHERE
    t.is_cycle = FALSE           -- stop once cycle is found
    AND t.next_id IS NOT NULL    -- stop at end of chain
    AND t.depth < 1000           -- safety cap
)
SELECT
  start_id,
  current_id        AS cycle_entry_node,
  depth,
  visited_path
FROM traversal
WHERE is_cycle = TRUE;
```

**How the visited path works:**
- Anchor: `visited_path = '1'` (just the start node)
- After step 1: `'1,2'`
- After step 2: `'1,2,3'`
- Step 3 tries to visit node 1 again: `FIND_IN_SET(1, '1,2,3') > 0` → cycle detected, stop

**FIND_IN_SET vs INSTR for cycle guard:**
| Method | Use when |
| :--- | :--- |
| `FIND_IN_SET(id, path)` | IDs are integers or short strings with no commas |
| `INSTR(path, CONCAT(',', id, ','))` | Need to avoid partial matches (e.g. ID 1 matching inside ID 12) |

```sql
-- Safer INSTR version that avoids partial ID matches
-- Wrap path with commas on both ends so every ID is surrounded by commas
INSTR(CONCAT(',', t.visited_path, ','), CONCAT(',', n.id, ',')) > 0
```

**Common mistakes:**
- No safety depth cap — if the cycle guard has a bug, the query runs forever; always add `AND depth < N`
- Using `FIND_IN_SET` with string IDs that contain commas — use `INSTR` with the comma-wrap trick instead
- Starting only from one anchor node — cycles that don't include the anchor node won't be detected; start from all nodes or the known entry point

---
### §8g — Cycle length measurement

**Signal:** "how long is the cycle", "size of the loop", "number of nodes in the cycle"

Once you detect a cycle (§8f), the cycle length is the number of steps from when a node was first visited to when it appears again. This equals `depth - entry_depth`, where `entry_depth` is the depth at which the repeated node first appeared in the path.

```sql
-- Table: nodes(id, next_id)
WITH RECURSIVE traversal AS (
  SELECT
    id                          AS start_id,
    id                          AS current_id,
    next_id,
    CAST(id AS CHAR(1000))      AS visited_path,
    0                           AS depth
  FROM nodes
  WHERE id = 1                  -- start from a specific node

  UNION ALL

  SELECT
    t.start_id,
    n.id,
    n.next_id,
    CONCAT(t.visited_path, ',', n.id),
    t.depth + 1
  FROM nodes n
  JOIN traversal t ON n.id = t.next_id
  WHERE
    FIND_IN_SET(n.id, t.visited_path) = 0   -- keep going until cycle
    AND t.next_id IS NOT NULL
    AND t.depth < 1000
),
-- The last row written before the guard fired is the row just BEFORE revisiting
-- The revisited node's first appearance depth tells us where the cycle started
cycle_entry AS (
  SELECT
    t.current_id            AS last_node,
    t.next_id               AS repeated_node,
    t.depth                 AS cycle_close_depth,
    -- find the depth at which repeated_node first appeared in the path
    -- count commas before it in the path string as a proxy for its depth
    LENGTH(
      SUBSTRING_INDEX(t.visited_path, CONCAT(',', t.next_id), 1)
    ) - LENGTH(
      REPLACE(SUBSTRING_INDEX(t.visited_path, CONCAT(',', t.next_id), 1), ',', '')
    )                       AS entry_depth
  FROM traversal t
  WHERE t.next_id IS NOT NULL
    AND FIND_IN_SET(t.next_id, t.visited_path) > 0
  ORDER BY t.depth DESC
  LIMIT 1
)
SELECT
  repeated_node             AS cycle_start_node,
  cycle_close_depth - entry_depth AS cycle_length
FROM cycle_entry;
```

**Simpler alternative — count nodes in the cycle by re-traversing it:**

```sql
-- Once you know the cycle entry node (from §8f), walk the cycle and count steps
WITH RECURSIVE cycle_walk AS (
  SELECT
    id          AS current_id,
    next_id,
    1           AS steps
  FROM nodes
  WHERE id = <cycle_entry_node>   -- substitute the detected entry node

  UNION ALL

  SELECT
    n.id,
    n.next_id,
    cw.steps + 1
  FROM nodes n
  JOIN cycle_walk cw ON n.id = cw.next_id
  WHERE n.id != <cycle_entry_node>  -- stop when we return to the entry node
    AND cw.steps < 1000
)
SELECT MAX(steps) AS cycle_length FROM cycle_walk;
```

**Which approach to use:**
| Approach | Use when |
| :--- | :--- |
| Path-string depth subtraction | Single pass; useful when you also need the entry node |
| Re-traverse from entry node | Cleaner and more readable; use when entry node is already known |

---
### §8h — Shortest path between two nodes (BFS)

**Signal:** "shortest path", "minimum hops", "fewest steps to reach", "minimum edges between nodes"

A recursive CTE naturally implements BFS (breadth-first search) because it expands all nodes at depth 1 before depth 2, and so on. The first time the target node is reached, that depth is the shortest path length. The visited path guard is **per path, not global**: each row carries its own `visited` string, so it stops a branch from looping back on itself, but separate branches cannot see each other's visited sets and the same node may be expanded on several branches. The result is still correct — `MIN(depth)` picks the shortest — but the work grows quickly on dense graphs.

```sql
-- Table: edges(from_node, to_node)  -- directed graph; add both directions for undirected
WITH RECURSIVE bfs AS (
  -- Anchor: first hop out of the start node, so depth starts at 1
  SELECT
    from_node                   AS current_node,
    to_node                     AS next_node,
    1                           AS depth,
    CAST(from_node AS CHAR(1000)) AS visited
  FROM edges
  WHERE from_node = <start_node>

  UNION ALL

  SELECT
    e.from_node,
    e.to_node,
    b.depth + 1,
    CONCAT(b.visited, ',', e.from_node)
  FROM edges e
  JOIN bfs b ON e.from_node = b.next_node
  WHERE
    FIND_IN_SET(e.from_node, b.visited) = 0   -- don't revisit nodes
    AND b.next_node != <target_node>           -- stop expanding once target is found
    AND b.depth < 100                          -- safety cap
)
SELECT MIN(depth) AS shortest_path
FROM bfs
WHERE next_node = <target_node>;
```

**Return the full path, not just the length:**

```sql
WITH RECURSIVE bfs AS (
  SELECT
    from_node                             AS current_node,
    to_node                               AS next_node,
    1                                     AS depth,
    CAST(from_node AS CHAR(1000))         AS visited,
    CAST(CONCAT(from_node, '->', to_node) AS CHAR(1000)) AS path_str
  FROM edges
  WHERE from_node = <start_node>

  UNION ALL

  SELECT
    e.from_node,
    e.to_node,
    b.depth + 1,
    CONCAT(b.visited, ',', e.from_node),
    CONCAT(b.path_str, '->', e.to_node)
  FROM edges e
  JOIN bfs b ON e.from_node = b.next_node
  WHERE
    FIND_IN_SET(e.from_node, b.visited) = 0
    AND b.next_node != <target_node>
    AND b.depth < 100
)
SELECT depth AS shortest_path, path_str
FROM bfs
WHERE next_node = <target_node>
ORDER BY depth
LIMIT 1;
```

**Undirected graph:** add both directions to the edges table, or UNION both directions in the CTE:
```sql
-- Treat every edge as bidirectional
WITH all_edges AS (
  SELECT from_node, to_node FROM edges
  UNION ALL
  SELECT to_node, from_node FROM edges   -- reverse direction
)
-- then use all_edges instead of edges in the BFS CTE above
```

**Common mistakes:**
- Forgetting the visited guard — without it, BFS loops on cycles and never terminates
- Stopping expansion too early — the `next_node != target` condition must go in `WHERE`, not as a join condition, or you'll miss the target row
- Using directed edges for an undirected problem — always UNION the reverse direction

---
### §8i — Running total / cost accumulation through a chain

**Signal:** "bill of materials", "total cost from root to node", "accumulated value down a hierarchy", "running sum through a chain"

When each node has a cost/weight and you need the total cost from root to each node, carry a running accumulator in the recursive step.

```sql
-- Table: bom(component_id, parent_id, unit_cost)
-- Goal: total cost from root to each node (sum of unit_cost along the path)
WITH RECURSIVE cost_tree AS (
  -- Anchor: root components (no parent)
  SELECT
    component_id,
    parent_id,
    unit_cost,
    unit_cost           AS total_cost,   -- running total starts at own cost
    0                   AS depth,
    CAST(component_id AS CHAR(500)) AS path
  FROM bom
  WHERE parent_id IS NULL

  UNION ALL

  SELECT
    b.component_id,
    b.parent_id,
    b.unit_cost,
    ct.total_cost + b.unit_cost,         -- accumulate down the tree
    ct.depth + 1,
    CONCAT(ct.path, '->', b.component_id)
  FROM bom b
  JOIN cost_tree ct ON b.parent_id = ct.component_id
)
SELECT component_id, depth, total_cost, path
FROM cost_tree
ORDER BY path;
```

**Accumulation patterns — what you can carry:**

| Goal | Accumulator in anchor | Accumulator in recursive step |
| :--- | :--- | :--- |
| Sum of costs | `unit_cost AS total_cost` | `ct.total_cost + b.unit_cost` |
| Product (e.g. probability) | `prob AS running_prob` | `ct.running_prob * b.prob` |
| Concatenated path labels | `CAST(name AS CHAR(500))` | `CONCAT(ct.path, ' > ', b.name)` |
| Max cost along path | `unit_cost AS max_cost` | `GREATEST(ct.max_cost, b.unit_cost)` |
| Depth / level | `0 AS depth` | `ct.depth + 1` |

**Multiply quantity down BOM levels:**

```sql
-- Each node has quantity_per_parent; total quantity = product of quantities from root
WITH RECURSIVE bom_qty AS (
  SELECT component_id, parent_id, qty_per_parent,
    1                   AS total_qty   -- root has no parent, so the multiplier starts at 1
  FROM bom
  WHERE parent_id IS NULL

  UNION ALL

  SELECT b.component_id, b.parent_id, b.qty_per_parent,
    bq.total_qty * b.qty_per_parent    -- multiply quantities down the chain
  FROM bom b
  JOIN bom_qty bq ON b.parent_id = bq.component_id
)
SELECT component_id, total_qty FROM bom_qty;
```

**Common mistakes:**
- Forgetting to initialize the accumulator in the anchor — if the anchor omits `total_cost`, the recursive step has nothing to add to
- Using `SUM()` in the recursive step — you can't use aggregate functions inside the recursive part of a CTE; carry the running value explicitly instead
- Not casting the path string to a long enough `CHAR` — MySQL truncates `CAST(... AS CHAR(n))` silently; use at least 500-1000 chars

---
### §8j — Flatten a linked list

**Signal:** "linked list", "each row has a next_id pointer", "kth node in a chain", "reorder by chain position", "find the tail"

A linked list table has each row pointing to the next via a `next_id` column (NULL = tail). The recursive CTE walks from the head and assigns a sequential position to each node.

```sql
-- Table: list_nodes(id, val, next_id)
-- Find the head: the node that is NOT someone else's next_id
WITH head AS (
  SELECT id FROM list_nodes
  WHERE id NOT IN (
    SELECT next_id FROM list_nodes WHERE next_id IS NOT NULL
  )
),
-- Walk the chain from head, assign position 1, 2, 3, ...
WITH RECURSIVE flattened AS (
  SELECT
    ln.id,
    ln.val,
    ln.next_id,
    1 AS position
  FROM list_nodes ln
  JOIN head h ON ln.id = h.id

  UNION ALL

  SELECT
    ln.id,
    ln.val,
    ln.next_id,
    f.position + 1
  FROM list_nodes ln
  JOIN flattened f ON ln.id = f.next_id
)
SELECT id, val, position FROM flattened ORDER BY position;
```

**Common tasks once the list is flattened:**

```sql
-- Find the kth node
SELECT val FROM flattened WHERE position = k;

-- Find the tail
SELECT val FROM flattened WHERE next_id IS NULL;

-- Find the middle node (ceiling for even-length lists)
SELECT val FROM flattened
WHERE position = CEIL((SELECT COUNT(*) FROM flattened) / 2.0);

-- Reverse the list: output in reverse position order
SELECT id, val FROM flattened ORDER BY position DESC;
```

**Flatten with two-pointer trick (find middle in one pass):**

```sql
-- Carry both a slow pointer (advances 1) and a fast pointer (advances 2)
-- When the fast pointer reaches the end, slow is at the middle
WITH RECURSIVE two_ptr AS (
  SELECT
    ln.id       AS slow_id,
    ln.next_id  AS slow_next,
    ln.id       AS fast_id,
    ln.next_id  AS fast_next
  FROM list_nodes ln
  JOIN head h ON ln.id = h.id

  UNION ALL

  SELECT
    s.id        AS slow_id,
    s.next_id   AS slow_next,
    f2.id       AS fast_id,
    f2.next_id  AS fast_next
  FROM two_ptr tp
  JOIN list_nodes s  ON s.id  = tp.slow_next
  JOIN list_nodes f1 ON f1.id = tp.fast_next     -- fast advances once
  JOIN list_nodes f2 ON f2.id = f1.next_id       -- fast advances twice
  WHERE tp.fast_next IS NOT NULL
    AND f1.next_id  IS NOT NULL
)
SELECT slow_id AS middle_node_id FROM two_ptr
ORDER BY fast_id DESC
LIMIT 1;
```

**Common mistakes:**
- Finding the head with `WHERE next_id IS NULL` — that finds the *tail*, not the head; the head is the node that appears in no other row's `next_id`
- Forgetting `WHERE next_id IS NOT NULL` in the head subquery — a NULL `next_id` in the subquery makes `NOT IN` return empty (same NULL trap as anti-joins)
- Not adding a depth/safety cap — if the data has a cycle, the CTE runs forever; add `AND position < 10000`

---
## §9 — Advanced Aggregation Patterns

**Signal:** "best match", "mapping", "all rows must meet condition", "pair-based comparison", "tuple condition"

### §9a — Mapping / best-match with GROUP BY + MIN/MAX

**Signal:** "find the best matching X for each Y", "assign the closest", "match records by criteria"

When you need to find the best match between two sets of records, set up the join criteria first, then use `GROUP BY` + `MIN` or `MAX` to select the single best match per group.

```sql
-- Find the closest assignment (smallest gap to the preferred date) for each student
WITH matches AS (
  SELECT
    s.student_id,
    a.assignment_id,
    a.due_date,
    ABS(DATEDIFF(s.preferred_date, a.due_date)) AS date_diff
  FROM students s
  JOIN assignments a
    ON a.course_id = s.course_id           -- eligibility condition
    AND a.due_date >= s.enrollment_date    -- additional filter
)
SELECT student_id, MIN(date_diff) AS closest_gap
FROM matches
GROUP BY student_id;
-- Then rejoin to get the full assignment row if needed
```

### §9b — Tuple / pair conditions

**Signal:** "find rows where (col_a, col_b) matches a known pair"

Use tuple conditions instead of two separate `AND` filters when checking compound key matches.

```sql
-- Check if (product_id, warehouse_id) pair exists in a list
SELECT * FROM inventory
WHERE (product_id, warehouse_id) IN (
  SELECT product_id, warehouse_id FROM orders WHERE status = 'pending'
);

-- Self-join using tuple to find symmetric pairs
SELECT a.user1_id, a.user2_id
FROM friendships a
JOIN friendships b
  ON (a.user1_id, a.user2_id) = (b.user2_id, b.user1_id);  -- bidirectional check
```

### §9c — "All rows must meet condition" (full coverage check)

**Signal:** "only include groups where every row satisfies X", "mandatory requirement for all"

Use conditional aggregation to count how many rows satisfy the condition and compare to the total.

```sql
-- Only include students who passed ALL mandatory courses
SELECT student_id
FROM grades
WHERE mandatory = 'Yes'
GROUP BY student_id
HAVING
  SUM(CASE WHEN grade = 'A' THEN 1 ELSE 0 END) = COUNT(*);
  -- WHERE already restricts the rows to mandatory courses, so COUNT(*) is the
  -- total mandatory count; if the grade-A count matches it, all of them passed.
  -- Keep the SUM(CASE WHEN mandatory...) form only if you drop the WHERE clause.
```

### §9d — Multiple CTEs as staged filters

**Signal:** complex eligibility rules, multi-step funnel, layered conditions

Build up the answer step-by-step: each CTE applies one layer of filtering or transformation.

```sql
WITH step1_eligible AS (
  -- Filter to active users only
  SELECT user_id FROM users WHERE status = 'active'
),
step2_qualified AS (
  -- Among active users, those who completed required courses
  SELECT g.student_id
  FROM grades g
  JOIN step1_eligible e ON g.student_id = e.user_id
  WHERE g.mandatory = 'Yes'
  GROUP BY g.student_id
  HAVING COUNT(*) = SUM(CASE WHEN g.grade = 'A' THEN 1 ELSE 0 END)
),
step3_ranked AS (
  -- Rank the qualified students by GPA
  SELECT s.student_id, s.gpa,
    DENSE_RANK() OVER (ORDER BY s.gpa DESC) AS gpa_rank
  FROM students s
  JOIN step2_qualified q ON s.student_id = q.student_id
)
SELECT * FROM step3_ranked WHERE gpa_rank <= 10;
```

### §9e — Row-level vs aggregate mixing (avoid or resolve)

**Signal:** you need a value like `COUNT(*)` from the full table alongside row-level data

You cannot mix row-level and aggregate values in the same `SELECT` without `GROUP BY`. If you need aggregate context for each row, either use a window function or join back to a CTE.

```sql
-- WRONG: mixing row-level and aggregate
SELECT user_id, COUNT(*) FROM orders;  -- error or single-row result

-- RIGHT option 1: window function
SELECT user_id, order_id,
  COUNT(*) OVER () AS total_orders   -- window function keeps all rows
FROM orders;

-- RIGHT option 2: join back to aggregate CTE
WITH totals AS (
  SELECT COUNT(*) AS total_orders FROM orders
)
SELECT o.user_id, o.order_id, t.total_orders
FROM orders o
CROSS JOIN totals t;
```

**Common mistakes:**
- Using a self-join without `ON a.id < b.id` — creates duplicate pairs (a,b) and (b,a)
- Tuple conditions with NULLs — `(a, NULL) IN (...)` behaves unexpectedly; filter NULLs first
- Mixing row-level and aggregate values without a window function or subquery

---
# Decision Guide

```
What is the problem asking?
│
├── Remove or identify duplicate rows?
│   ├── All columns identical                → DISTINCT
│   ├── Deduplicate + aggregate              → GROUP BY
│   ├── Keep first / last / nth occurrence   → ROW_NUMBER() + filter
│   ├── Find rows with no match elsewhere    → NOT EXISTS
│   └── Physically delete duplicates         → DELETE + self-join
│
├── Find top N or ranked rows?
│   ├── Exactly N rows, ties broken          → ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)
│   ├── Top N values, ties all included      → DENSE_RANK() OVER (...) WHERE dr <= N
│   └── Single Nth value                     → DENSE_RANK or LIMIT + OFFSET (prefer DENSE_RANK)
│
├── Pivot rows into columns?
│   ├── Fixed, known columns                 → SUM(CASE WHEN col = val THEN ... END)
│   ├── Top N items as columns               → ROW_NUMBER() + MAX(CASE WHEN rn = k THEN ...)
│   └── Dynamic columns                      → Dynamic SQL with GROUP_CONCAT
│
├── Find consecutive sequences or gaps?
│   ├── Fixed step, dates in MySQL           → DATE_SUB(date, INTERVAL ROW_NUMBER() DAY)
│   ├── Fixed step, integers (any dialect)   → value - ROW_NUMBER() grouping trick
│   ├── Condition-based boundary             → flag is_new_group + cumulative SUM
│   ├── Just checking for gaps               → LAG comparison + TIMESTAMPDIFF
│   ├── Consecutive months                   → PERIOD_DIFF on DATE_FORMAT '%Y%m'
│   └── Find missing values                  → Recursive date series + LEFT JOIN IS NULL
│
├── Product / user metrics?
│   ├── Active users in a fixed window       → COUNT(DISTINCT) with INTERVAL filter
│   ├── Active users in a sliding window     → Self-join with BETWEEN date range
│   ├── Cohort retention                     → cohort CTE + DATEDIFF / 7 for week_num
│   ├── Funnel (unordered steps)             → COUNT(DISTINCT CASE WHEN step = X)
│   ├── Funnel (ordered steps)               → LEFT JOIN ... ON step + event_time order
│   └── WoW / MoM growth                    → aggregate first in CTE, then LAG
│
├── Hierarchical or recursive data?
│   ├── Top-down traversal                   → Recursive CTE, anchor = root
│   ├── Bottom-up (find ancestors)           → Recursive CTE, anchor = leaf node
│   ├── Path / depth tracking                → Include path/lvl column in anchor + recursive step
│   ├── Generate date / number series        → Recursive CTE, anchor = start value
│   ├── Detect a cycle in a graph            → Carry visited_path; stop when FIND_IN_SET > 0 (§8f)
│   ├── Measure cycle length                 → Re-traverse from entry node; count steps (§8g)
│   ├── Shortest path / minimum hops         → BFS with visited guard; MIN(depth) at target (§8h)
│   ├── Accumulate cost / value down chain   → Carry running total in recursive step (§8i)
│   └── Flatten a linked list / find kth     → Walk from head via next_id; assign position (§8j)
│
├── Window function frame?
│   ├── Last k physical rows                 → ROWS BETWEEN k PRECEDING AND CURRENT ROW
│   ├── All rows in partition                → ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
│   ├── Value-based / tie-aware              → RANGE BETWEEN ... AND CURRENT ROW
│   ├── Auto-fill missing time periods       → RANGE BETWEEN (fills gaps with 0 automatically)
│   └── Running total                        → ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
│
├── Advanced aggregation?
│   ├── Best match / closest record          → JOIN on eligibility, GROUP BY + MIN/MAX
│   ├── All rows in group must pass          → HAVING SUM(condition) = SUM(mandatory)
│   ├── Compound key pair matching           → (col_a, col_b) IN (subquery)
│   ├── Row-level + aggregate together       → Window function or CROSS JOIN to CTE
│   └── Multi-step eligibility               → Chained CTEs, one filter per step
│
├── String matching?
│   ├── Simple contains / starts / ends      → LIKE with % and _
│   ├── Exact substring replacement          → REPLACE()
│   ├── Format validation / complex pattern  → REGEXP / REGEXP_LIKE
│   ├── Extract portion matching pattern     → REGEXP_SUBSTR
│   └── Known position or delimiter          → SUBSTRING / LEFT / RIGHT / INSTR
│
└── Deduplication inside aggregation?
    ├── Count unique values                  → COUNT(DISTINCT col)
    └── Sum only matching rows               → SUM(CASE WHEN ... THEN val ELSE 0 END)
```
